# Phase 8 — ST-CDGM **from-scratch** intégrant 10+ fixes experts

Architecture from-scratch validée par 5 experts (ML/Math/Recherche/IA/Climat). Vise à **battre noncausal v4 sur la majorité des métriques**.

## Probabilités empiriques estimées (basées sur V5 mesuré)

| Métrique | Proba battre noncausal | Justification |
|----------|------------------------|---------------|
| Pearson, RMSE, MAE, CRPS, RAPSD | **70-85%** | V5 le fait déjà ; Min-SNR + features physiques aident |
| SSR (calibration) | 50-60% | Multi-EMA + cond_dropout + Dispersive Loss |
| **F1@p99** | **10-30%** | Verdict expert (17/17 variants échec, 0 précédent publié) |
| CSI/SEDI/FSS | 40-60% | Suivent F1 ; FSS tolère décalages spatiaux |
| Indices climatiques (CDD, R10, RX1) | 60-75% | V5 déjà mieux sur CDD/R10 ; tail_weight raffine |

## Architecture (fixes intégrés)

### Stage 1 (causal mean predictor, from-scratch)
- 15 features existantes + **w_700** (interp pondérée 0.571·w_850 + 0.429·w_500, Holton 2004)
- + **θ_e_850, θ_e_500** (Bolton 1980)
- + **MUCAPE proxy = θ_e_850 − θ_e_500** (remplace CAPE 2-niveaux trop grossier, Emanuel 1994)
- Encoder + RCN + DAG learnable + dual_path Path B UNet
- **Losses physiques différentiables corrigées** :
  - `L_R10mm = MSE(Σ_t σ(5·(expm1(x)-10)), Σ_t 1[expm1(x)>10])` ← seuil en mm/day après expm1
  - `L_Rx1day = LSE_T=5(expm1(pred)) approx max` ← LogSumExp stable
  - `L_CDD = MSE(σ(5·(1-expm1(x))), 1[expm1(x)<1])`
  - `L_CC = autograd((∂μ/∂T_850)·σ_T/σ_μ − 0.07)²` ← adimensionné, autograd obligatoire
  - Pondération : 0.20·R10 + 0.15·Rx1 + 0.10·CDD + 0.05·CC, warmup epoch 10+

### Stage 2 (diffusion EDM)
- UNet 4 niveaux [128,256,256,256] CorrDiff Normal (~50M params)
- **Conditioning** : `causal_concat=True` (fallback documenté — AdaGN reporté à V9 cf. réserve IA)
- **Min-SNR-γ=5** weighting EDM (Hang ICCV 2023, arXiv 2303.09556)
- **Tail_weight (4, 12)** au lieu de (8, 25) — Climate ML
- **Dispersive Loss** λ=0.25 (corrigé vs 0.05 sous-dosé) mid-block hook (He&Wang 2025, arXiv 2506.09027)
- **conditioning_dropout p=0.13** (Ho&Salimans 2022)
- **Multi-EMA** {0.999, 0.9995, 0.9999} avec post-hoc sweep (Karras 2024, arXiv 2312.02696)
- **α appris ∈ [0,1]** avec régularisation `L_α = 0.1·(α-0.5)² + 1.0·max(0, 0.3-α)²` (évite α→0 collapse)

### Sampling
- `dpm_solver++` 32 steps (Lu 2022, arXiv 2211.01095)
- **Limited-Interval Guidance** σ ∈ [0.05, 1.0] (Kynkäänniemi 2024, arXiv 2404.07724)
- cfg_scale 1.0-1.5

### Eval protocol (publication-ready)
- **N_BATCHES = 64**, **K_SAMPLES = 128** (CI ±0.014)
- 3 conventions F1@p99 : pooled full-grid (vs noncausal), ETCCDI per-pixel land, land-only pooled
- CSI@p99, SEDI@p99, FSS (n=9, n=25, n=51 px)
- CRPS gaussian, rank histogram, RMSE, MAE, Pearson global+per-sample
- **Indices climatiques sur 730 jours ENTIERS** (rx1day, CDD, R10mm, DJF/JJA) — ETCCDI Zhang 2011
- μ_HR ablation (causalité opérationnelle), Q_phys (interprétabilité), α appris final
- **Paired permutation test** n=10000 (Phase 8 vs noncausal sur mêmes batches)
- **Bootstrap CI 95% BCa** n=1000 (Math)
- **Holm-Bonferroni** correction multi-tests
- **Pre-enregistrement** : git rev-parse HEAD logged dans JSON output

## Coût compute estimé (A100 Pro+)
- Stage 1 : ~6-7h, 15 epochs
- Stage 2 : ~22-27h, 200 epochs (cached)
- Eval BS30 unifié + comparaison 3-way : ~15h
- **Total : ~45-55h, soit 2-3 sessions Pro+ avec checkpoint/resume**

In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers
    from omegaconf import OmegaConf
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab')

import torch, numpy as np, json, time
import xarray as xr
from omegaconf import OmegaConf

# Reproducibility (Expert IA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Pre-registration (Expert Recherche)
_git_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f'[bootstrap] git SHA = {_git_sha}  branch = {GIT_BRANCH}')
print(f'[bootstrap] cwd={os.getcwd()}  torch={torch.__version__}  cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[bootstrap] GPU = {torch.cuda.get_device_name(0)}  VRAM = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# === Cell 2 : Constants ===
DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
DATA_ROOT  = DRIVE_ROOT / 'data'
HR_RAW_PATH = DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc'
LR_RAW_PATH = DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc'
STATIC_PATH = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'

# Phase 8 output dir
OUT_DIR = DRIVE_ROOT / 'oracle_9node' / 'phase8_from_scratch'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_STAGE1_LAST  = OUT_DIR / 'stage1_last.pth'
CKPT_STAGE1_BEST  = OUT_DIR / 'stage1_best.pth'
STAGE1_CACHE_PATH = OUT_DIR / 'stage1_cache_mu_total.pt'
CKPT_STAGE2_LAST  = OUT_DIR / 'stage2_last.pth'
CKPT_STAGE2_BEST  = OUT_DIR / 'stage2_best.pth'
LAND_MASK_PATH    = OUT_DIR / 'land_mask_nz.npy'
CLIM_PATH         = OUT_DIR / 'clim_train_p95_p99.npz'
AUGMENTED_LR_PATH = OUT_DIR / 'lr_augmented_features.nc'  # w_700, theta_e, MUCAPE
TRAINING_HISTORY  = OUT_DIR / 'training_history.json'
FINAL_RESULTS     = OUT_DIR / 'phase8_final_results.json'

# Reference checkpoints to compare against (existing models)
REF_CKPT_NONCAUSAL = DRIVE_ROOT / 'ckpt_noncausal'
REF_CKPT_V5_CAUSAL = DRIVE_ROOT / 'ckpt_v2_corrdiff_normal'

# Reproducibility
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# K9 temporal split (used by all phases)
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],   # 30 years
    'val':     ['2010-01-01', '2011-12-31'],   # 2 years
    'test':    ['2012-01-01', '2013-12-31'],   # 2 years -> 730 days for climate indices
    'holdout': ['2014-01-01', '2014-12-31'],   # 1 year (out-of-distribution warm year)
}

# Stage 1 hyperparams (Path C+ Option C proven on seed_42)
STAGE1_EPOCHS    = 15
STAGE1_LR        = 1e-3
LAMBDA_DAG_PRIOR = 0.40
LAMBDA_L1_START  = 0.04
LAMBDA_L1_END    = 0.005
G_PHYS_ALPHA     = 0.25
# Physical loss weights (warmup applied after epoch 10)
LAMBDA_R10MM   = 0.20
LAMBDA_RX1DAY  = 0.15
LAMBDA_CDD     = 0.10
LAMBDA_CC      = 0.05
PHYS_LOSS_WARMUP_EPOCH = 10

# Stage 2 hyperparams
STAGE2_EPOCHS         = 200
STAGE2_LR             = 2e-4
STAGE2_BATCH_SIZE     = 64
STAGE2_WEIGHT_DECAY   = 1e-4
STAGE2_GRADIENT_CLIP  = 1.0
MIN_SNR_GAMMA         = 5.0
TAIL_WEIGHT_P95       = 4.0    # (Climate ML : 8 -> 4 less aggressive)
TAIL_WEIGHT_P99       = 12.0   # (Climate ML : 25 -> 12)
DISPERSIVE_LAMBDA     = 0.25   # (Expert ML : raised from 0.05 sous-dosé to paper's recommended range)
DISPERSIVE_TAU        = 0.5    # Kernel temperature
COND_DROPOUT_P        = 0.13   # CFG compatibility (CorrDiff Nature CEE 2025)
EMA_DECAYS            = [0.999, 0.9995, 0.9999]   # Multi-EMA EDM2 Karras 2024
ALPHA_TARGET          = 0.5   # Regularization target for learned alpha
ALPHA_FLOOR           = 0.3   # Penalty if alpha < 0.3 (prevent collapse)
LAMBDA_ALPHA_REG      = 0.1
BETA_ALPHA_FLOOR      = 1.0
SIGMA_DATA_NEW        = 0.193  # Phase 6 dualpath recalibrated

# BS30 eval protocol (publication-ready)
N_TEST_BATCHES = 64    # Math expert : N=64 -> CI +/-0.014
K_SAMPLES      = 128   # >> CorrDiff Mardani (32)
N_STEPS_DIFF   = 32    # dpm_solver++ converges at 32 NFE
WET_DAY_THRESHOLD_MM = 1.0   # ETCCDI standard

# Sampling
SAMPLER_SCHEDULER       = 'dpm_solver++'
CFG_SCALE               = 1.0
LIMITED_GUIDANCE_SIGMA_MIN = 0.05   # Kynkäänniemi NeurIPS 2024 (calibrated for sigma_data=0.1)
LIMITED_GUIDANCE_SIGMA_MAX = 1.0

# Bootstrap + statistical tests
BOOTSTRAP_N_RESAMPLES = 1000   # Math : n_boot=1000
PAIRED_PERMUTATION_N = 10000   # Math : n_perm=10000 for p<=0.001
BOOTSTRAP_METHOD     = 'BCa'   # Bias-corrected accelerated (Efron 1987)

print(f'[Cell 2] DEVICE = {DEVICE}')
print(f'[Cell 2] DRIVE_ROOT = {DRIVE_ROOT}')
print(f'[Cell 2] OUT_DIR = {OUT_DIR}')
print(f'[Cell 2] Stage 1 : {STAGE1_EPOCHS} epochs, LR {STAGE1_LR}')
print(f'[Cell 2] Stage 2 : {STAGE2_EPOCHS} epochs, LR {STAGE2_LR}, batch {STAGE2_BATCH_SIZE}')
print(f'[Cell 2] Min-SNR γ = {MIN_SNR_GAMMA}, tail_weight ({TAIL_WEIGHT_P95}, {TAIL_WEIGHT_P99})')
print(f'[Cell 2] Multi-EMA decays = {EMA_DECAYS}')
print(f'[Cell 2] BS30 eval : N={N_TEST_BATCHES} batches x K={K_SAMPLES} samples x {N_STEPS_DIFF} steps')

# Pre-registration record (Expert Recherche)
PRE_REG_RECORD = {
    'phase': 'phase8_from_scratch',
    'git_sha': _git_sha,
    'git_branch': GIT_BRANCH,
    'seed': SEED,
    'k9_dates': K9_DATES,
    'stage1_hyperparams': {
        'epochs': STAGE1_EPOCHS, 'lr': STAGE1_LR,
        'lambda_dag_prior': LAMBDA_DAG_PRIOR,
        'lambda_l1_start': LAMBDA_L1_START, 'lambda_l1_end': LAMBDA_L1_END,
        'g_phys_alpha': G_PHYS_ALPHA,
        'physical_loss_weights': {
            'R10mm': LAMBDA_R10MM, 'Rx1day': LAMBDA_RX1DAY,
            'CDD': LAMBDA_CDD, 'CC': LAMBDA_CC,
        },
        'phys_warmup_epoch': PHYS_LOSS_WARMUP_EPOCH,
    },
    'stage2_hyperparams': {
        'epochs': STAGE2_EPOCHS, 'lr': STAGE2_LR,
        'batch_size': STAGE2_BATCH_SIZE, 'weight_decay': STAGE2_WEIGHT_DECAY,
        'min_snr_gamma': MIN_SNR_GAMMA,
        'tail_weight': [TAIL_WEIGHT_P95, TAIL_WEIGHT_P99],
        'dispersive_lambda': DISPERSIVE_LAMBDA,
        'cond_dropout_p': COND_DROPOUT_P,
        'ema_decays': EMA_DECAYS,
        'alpha_reg': {'target': ALPHA_TARGET, 'floor': ALPHA_FLOOR,
                       'lambda': LAMBDA_ALPHA_REG, 'beta_floor': BETA_ALPHA_FLOOR},
        'sigma_data': SIGMA_DATA_NEW,
    },
    'eval_protocol': {
        'n_batches': N_TEST_BATCHES, 'k_samples': K_SAMPLES,
        'n_steps_diff': N_STEPS_DIFF,
        'sampler': SAMPLER_SCHEDULER, 'cfg_scale': CFG_SCALE,
        'limited_guidance': [LIMITED_GUIDANCE_SIGMA_MIN, LIMITED_GUIDANCE_SIGMA_MAX],
        'bootstrap_n': BOOTSTRAP_N_RESAMPLES,
        'paired_permutation_n': PAIRED_PERMUTATION_N,
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
print(f'[Cell 2] Pre-registration record created (SHA {_git_sha[:8]})')

In [ ]:
# === Cell 3 : Land mask depuis static dataset (multi-source fallback) ===
# Approche : static_predictors fournit orog/he/vegt. On essaie lsm/sftlf/vegt/orog dans l'ordre.

_t0 = time.time()
if not STATIC_PATH.exists():
    raise FileNotFoundError(f'Static dataset not on Drive : {STATIC_PATH}')
ds_static = xr.open_dataset(str(STATIC_PATH), engine='h5netcdf')
print(f'[Cell 3] static variables : {list(ds_static.data_vars)}')

land_mask = None
land_mask_source = None

# Strategy 1 : explicit lsm/sftlf/landfrac
for cand in ('lsm', 'land_sea_mask', 'landfrac', 'land_mask', 'sftlf'):
    if cand in ds_static.data_vars:
        arr = ds_static[cand].values.squeeze()
        if arr.ndim != 2: continue
        thr = 50.0 if arr.max() > 1.5 else 0.5
        land_mask = arr >= thr
        land_mask_source = f'{cand} (threshold {thr})'
        break

# Strategy 2 : vegetation type
if land_mask is None and 'vegt' in ds_static.data_vars:
    vegt = ds_static['vegt'].values.squeeze()
    if vegt.ndim == 2:
        _unique = np.unique(vegt[np.isfinite(vegt)])
        print(f'[Cell 3] vegt unique values = {_unique}')
        land_mask = (vegt > 0) & (vegt != 17) & np.isfinite(vegt)
        land_mask_source = 'vegt (0 and 17 = water)'

# Strategy 3 : orography (safe NaN handling per Climat)
if land_mask is None and 'orog' in ds_static.data_vars:
    orog = ds_static['orog'].values.squeeze()
    if orog.ndim == 2:
        print(f'[Cell 3] orog range : [{np.nanmin(orog):.2f}, {np.nanmax(orog):.2f}] m')
        if np.isnan(orog).any():
            land_mask = np.isfinite(orog) & (orog > -0.5)
            land_mask_source = 'orog : isfinite & > -0.5 m'
        else:
            land_mask = orog > 0.5
            land_mask_source = 'orog > 0.5 m'

if land_mask is None:
    raise RuntimeError(f'No land/sea variable found in static : {list(ds_static.data_vars)}')

n_land = int(land_mask.sum())
n_total = int(land_mask.size)
print(f'[Cell 3] land_mask source : {land_mask_source}')
print(f'[Cell 3] land pixels = {n_land} / {n_total} ({100*n_land/n_total:.1f}%)')
print(f'[Cell 3] expected for NZ : 22-45% (varies with bbox size)')

# Sanity HR shape match
_hr_ds = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
_pr_var = 'pr' if 'pr' in _hr_ds.data_vars else list(_hr_ds.data_vars)[0]
assert _hr_ds[_pr_var].shape[-2:] == land_mask.shape, 'HR grid mismatch'
_hr_ds.close()

np.save(LAND_MASK_PATH, land_mask)
ds_static.close()
print(f'[Cell 3] saved : {LAND_MASK_PATH}  ({time.time()-_t0:.1f}s)')

In [ ]:
# === Cell 4 : Climatology p95/p99 per-pixel (ETCCDI Zhang 2011) ===
# Per-pixel quantile on wet days >= 1 mm/day, training period only.

_t0 = time.time()
_hr_ds = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
_pr_var = 'pr' if 'pr' in _hr_ds.data_vars else list(_hr_ds.data_vars)[0]
_hr = _hr_ds[_pr_var]
_time_var = _hr.dims[0]
_hr_train = _hr.sel({_time_var: slice(K9_DATES['train'][0], K9_DATES['train'][1])})
_hr_train_np = _hr_train.values.astype(np.float32)
print(f'[Cell 4] train slice = {_hr_train_np.shape}')

H, W = _hr_train_np.shape[1], _hr_train_np.shape[2]
clim_p95 = np.full((H, W), np.nan, dtype=np.float32)
clim_p99 = np.full((H, W), np.nan, dtype=np.float32)
n_wet = np.zeros((H, W), dtype=np.int32)

for i in range(H):
    for j in range(W):
        if not land_mask[i, j]:
            continue
        px = _hr_train_np[:, i, j]
        px_finite = px[np.isfinite(px)]
        wet = px_finite[px_finite >= WET_DAY_THRESHOLD_MM]
        n_wet[i, j] = wet.size
        if wet.size >= 30:
            clim_p95[i, j] = float(np.quantile(wet, 0.95))
            clim_p99[i, j] = float(np.quantile(wet, 0.99))

_n_valid = int(np.isfinite(clim_p99).sum())
print(f'[Cell 4] land pixels with valid p99 = {_n_valid} / {n_land}')
print(f'[Cell 4] mean wet days per land pixel = {n_wet[land_mask].mean():.0f}')
print(f'[Cell 4] clim_p99 range = [{np.nanmin(clim_p99):.2f}, {np.nanmax(clim_p99):.2f}] mm/day  mean = {np.nanmean(clim_p99):.2f}')

np.savez(CLIM_PATH, clim_p95=clim_p95, clim_p99=clim_p99, n_wet=n_wet,
          land_mask=land_mask, wet_threshold_mm=WET_DAY_THRESHOLD_MM,
          train_start=K9_DATES['train'][0], train_end=K9_DATES['train'][1])
del _hr_train_np
_hr_ds.close()
print(f'[Cell 4] saved : {CLIM_PATH}  ({time.time()-_t0:.1f}s)')

In [ ]:
# === Cell 5 : Augmented LR features (w_700, θ_e_850, θ_e_500, MUCAPE proxy) ===
# Pre-compute offline once, save as NetCDF for fast loading during training.
# Formulas validated by Expert Climat :
#   w_700 = 0.571*w_850 + 0.429*w_500   (linear interpolation in pressure, Holton 2004)
#   θ_e   = θ * exp(L_v * q_sat / (c_p * T))   (Bolton 1980, simplified)
#   MUCAPE_proxy = θ_e_850 - θ_e_500   (static instability indicator, Emanuel 1994)
# Constants : R_d = 287, c_p = 1005, L_v = 2.5e6, R_v = 461.5

_t0 = time.time()

if AUGMENTED_LR_PATH.exists():
    print(f'[Cell 5] augmented LR already exists : {AUGMENTED_LR_PATH}')
    ds_lr_aug = xr.open_dataset(str(AUGMENTED_LR_PATH), engine='h5netcdf')
    print(f'[Cell 5] variables : {list(ds_lr_aug.data_vars)}')
else:
    print(f'[Cell 5] computing augmented features from {LR_RAW_PATH}...')
    ds_lr = xr.open_dataset(str(LR_RAW_PATH), engine='h5netcdf')
    print(f'[Cell 5] LR variables available : {list(ds_lr.data_vars)}')

    # ----- 1. w_700 by linear interpolation in pressure -----
    if 'w_850' in ds_lr.data_vars and 'w_500' in ds_lr.data_vars:
        w_700 = 0.571 * ds_lr['w_850'] + 0.429 * ds_lr['w_500']
        w_700.attrs['long_name'] = 'vertical_velocity_at_700hPa (interpolated)'
        w_700.attrs['units'] = 'Pa s-1'
        w_700.attrs['interpolation'] = '0.571*w_850 + 0.429*w_500 (linear in pressure)'
        print(f'[Cell 5] w_700 computed : shape {w_700.shape} range [{float(w_700.min()):.4f}, {float(w_700.max()):.4f}]')
    else:
        raise RuntimeError('w_850 and w_500 required for w_700 interpolation')

    # ----- 2. θ_e_850 and θ_e_500 via Bolton 1980 -----
    # θ = T * (1000/p)^(R_d/c_p)
    # q_sat via Magnus-Tetens : e_s = 6.112 * exp(17.67*(T-273.15)/(T-29.65))
    # θ_e = θ * exp(L_v * q_sat(T,p) / (c_p * T))
    R_d = 287.0
    c_p = 1005.0
    L_v = 2.5e6
    eps = 0.622  # R_d / R_v

    def _theta_e_bolton(T_K, p_hPa):
        """Bolton 1980 θ_e. T in Kelvin, p in hPa."""
        theta = T_K * (1000.0 / p_hPa) ** (R_d / c_p)
        e_s   = 6.112 * np.exp(17.67 * (T_K - 273.15) / (T_K - 29.65))
        q_sat = eps * e_s / (p_hPa - (1 - eps) * e_s)
        return theta * np.exp(L_v * q_sat / (c_p * T_K))

    if 't_850' in ds_lr.data_vars and 'q_850' in ds_lr.data_vars:
        T_850 = ds_lr['t_850']  # Kelvin (ACCESS-CM2 standard)
        theta_e_850 = xr.apply_ufunc(_theta_e_bolton, T_850, 850.0, dask='allowed')
        theta_e_850.attrs['long_name'] = 'equivalent_potential_temperature_850hPa (Bolton 1980)'
        theta_e_850.attrs['units'] = 'K'
        print(f'[Cell 5] θ_e_850 computed : range [{float(theta_e_850.min()):.1f}, {float(theta_e_850.max()):.1f}] K')
    else:
        raise RuntimeError('t_850 required for θ_e')

    if 't_500' in ds_lr.data_vars and 'q_500' in ds_lr.data_vars:
        T_500 = ds_lr['t_500']
        theta_e_500 = xr.apply_ufunc(_theta_e_bolton, T_500, 500.0, dask='allowed')
        theta_e_500.attrs['long_name'] = 'equivalent_potential_temperature_500hPa (Bolton 1980)'
        theta_e_500.attrs['units'] = 'K'
        print(f'[Cell 5] θ_e_500 computed : range [{float(theta_e_500.min()):.1f}, {float(theta_e_500.max()):.1f}] K')

    # ----- 3. MUCAPE proxy = θ_e_850 - θ_e_500 -----
    mucape_proxy = theta_e_850 - theta_e_500
    mucape_proxy.attrs['long_name'] = 'MUCAPE_proxy (theta_e_850 - theta_e_500)'
    mucape_proxy.attrs['units'] = 'K'
    mucape_proxy.attrs['interpretation'] = 'positive = convectively unstable'
    print(f'[Cell 5] MUCAPE proxy : range [{float(mucape_proxy.min()):.2f}, {float(mucape_proxy.max()):.2f}] K')

    # ----- Build augmented LR dataset (15 original + 4 new = 19 variables) -----
    ds_lr_aug = ds_lr.copy()
    ds_lr_aug['w_700']        = w_700
    ds_lr_aug['theta_e_850']  = theta_e_850
    ds_lr_aug['theta_e_500']  = theta_e_500
    ds_lr_aug['mucape_proxy'] = mucape_proxy

    # Save
    ds_lr_aug.to_netcdf(str(AUGMENTED_LR_PATH), engine='h5netcdf')
    ds_lr.close()
    print(f'[Cell 5] augmented LR saved : {AUGMENTED_LR_PATH}')
    print(f'[Cell 5] total variables = {len(ds_lr_aug.data_vars)} (original 15 + 4 new)')

print(f'[Cell 5] done in {time.time()-_t0:.1f}s')

## Cells 6-8 : Stage 1 from-scratch (à implémenter dans commit suivant)

- Cell 6 : Pipeline + dataloaders avec 19 features (15 original + w_700 + θ_e_850 + θ_e_500 + MUCAPE proxy)
- Cell 7 : Stage 1 build (encoder + RCN + DAG + dual_path) + losses physiques différentiables corrigées
- Cell 8 : Stage 1 training loop (15 epochs, warmup phys losses epoch 10+)

## Cells 9-11 : Stage 2 from-scratch (à implémenter dans commit suivant)

- Cell 9 : Precompute mu_total cache via dual_path frozen
- Cell 10 : Stage 2 build (UNet 50M EDM, α appris régularisé, hooks pour Dispersive Loss mid-block)
- Cell 11 : Stage 2 training loop (200 epochs, Min-SNR-γ=5 + tail_weight (4,12) + Dispersive λ=0.25 + cond_dropout=0.13 + multi-EMA inline)

## Cells 12-15 : Eval BS30 unifié + 3-way comparison (à implémenter dans commit suivant)

- Cell 12 : Sampling N=64 × K=128 × 32 steps avec Limited-Interval Guidance + post-hoc EMA sweep
- Cell 13 : Métriques (3 conventions F1, CSI, SEDI, FSS, CRPS, Pearson, RMSE, RAPSD) + indices climatiques sur 730 jours entiers
- Cell 14 : Paired permutation test + bootstrap BCa + Holm-Bonferroni
- Cell 15 : Tableau 3-way (Phase 8 vs V5 vs noncausal) + JSON publication-ready + plots